## Cifrari di Feistel

Il DES è un cifrario iterato, ottenuto come caso particolare di un *cifrario di Feistel*. La struttura inventata da Feistel ha il vantaggio che la cifratura e la decifratura sono operazioni molto simili, spesso identiche, e che basta invertire il funzionamento del gestore della chiave per ottenere l'operazione inversa: quindi i circuiti di cifratura e decifratura sono spesso gli stessi.

In un cifrario di questo tipo ogni stato $u^i$ è diviso in due metà di uguale lunghezza chiamate $L^i$ e $R^i$. La funzione round $g$ è definita come segue: $g(L^{i-1},R^{i-1},K^i)=(L^i,R^i)$ dove

$$L^i=R^{i-1}$$

$$R^i=L^{i-1}\oplus f(R^{i-1},K^i)$$

Notiamo che la funzione $f$ è qualsiasi e non ha bisogno di soddisfare nessuna proprietà di ingettività, in quanto un round di tipo Feistel è sempre invertibile: una volta assegnata la chiave di round è sufficiente scambiare $L$ e $R$ e inserire le chiavi in ordine inverso

$$L^{i-1}=R^i \oplus f(L^i, K^i)$$

$$R^{i-1}=L^i.$$

-L'aumento della grandezza del blocco aumenta la sicurezza ma rallenta la velocità di cifratura e decifratura.

-L'aumento della lunghezza della chiave aumenta la sicurezza e rende la ricerca esaustiva più difficile ma rallenta l'algoritmo.

-L'aumento del numero di round aumenta la sicurezza ma rallenta l'algoritmo.

-L'aumento di complessità della funzione $f$ rende l'analisi più difficile ma rallenta l'algoritmo.

## Data encryption standard

Il DES è un cifrario di Feistel a 16 round i cui blocchi hanno lunghezza 64 bit: il cipherthext di lunghezza 64 bit è ottenuto criptando un plaintext $x$ della stessa lunghezza, e cifrato utilizzando una chiave di 56 bit.

Prima dei 16 round viene applicata al testo una permutazione iniziale $\textbf{IP}$ fissata. Denotiamo:

$$\textbf{IP}(x)=L^0R^0$$

Alla fine dei 16 round, invece, viene applicata la permutazione inversa, ottenendo il ciphertext $y$

$$y=\textbf{IP}^{-1}(R^{16},L^{16})$$

Descriviamo l'implementazione della **funzione f**:

La funzione

$$f:\{0,1\}^{32}\times \{0,1\}^{48}\rightarrow \{0,1\}^{32}$$

prende in input una stringa di 32 bit e una chiave di 48 bit.

Costruiamo il key schedule $(K^1,\dots,K^{16})$ che è costituito da chiavi da 48 bit derivate dalla chiave $K$ di 56 bit, ove ognuna di esse è ottenuta tramite una permutazione dei bit di $K$.

Denominati $A$ e $J$ rispettivamente il primo e secondo argomento di $f$, i passaggi per calcolare $f(A,J)$ sono i seguenti:

**1)** Viene utilizzata una *funzione di espansione* $E(A)$ per estendere $A$ ad una string di 48 bit, effettuando delle semplici operazioni di duplicazione;

**2)** Si calcola il valore di $E(A)\oplus J$ e il risultato viene espresso tramite la concatenazione di 8 stringhe da 6 bit $B=B_1B_2B_3B_4B_5B_6B_7B_8$;

**3)** Si applicano 8 $S$-box denotati $S_1, \dots, S_8$ dove ogni

$$S^i:\{0,1\}^6\rightarrow \{0,1\}^4.$$

Data una stringa di lunghezza 6 $B_j=b_1b_2b_3b_4b_5b_6$, $S_j(B_j)$ viene calcolata nel seguente modo: $b_1b_6$ sono la rappresentazione binaria di una riga $r$ di $S_j$ e i 4 bit $b_1b_2b_3b_4$ sono la rappresentazione binaria di una colonna $c$ di $S_j$ (ove $1\leq r \leq 3$, mentre $1\leq c \leq 15$). Allora $S_j(B_j)$ è data dalla rappresentazione binaria di $S_j(r,c)$ tramite una stringa di lunghezza 4. Poniamo $C_j=S_j(B_j)$ con $1 \leq j \leq 8$;

**4)** Si applica la permutazione $\textbf{P}$ alla stringa $$C=C_1C_2C_3C_4C_5C_6C_7C_8$$ di lunghezza 32 bit

La funzione di espansione $E$ è data dalla seguente tabella:

In [101]:
E=[32,1,2,3,4,5,4,5,6,7,8,9,8,9,10,11,12,13,12,13,14,15,16,17,16,17,18,19,20,21,20,21,22,23,24,25,24,25,26,27,28,29,28,29,30,31,32,1]

Data una stringa $A=(a_1,a_2,\dots,a_{32})$ allora $E(A)$ è data dalla seguente stringa

$$E(A)=(a_{32},a_1,a_2,a_3,a_4,a_5,a_4,\dots, a_{31},a_{32},a_1)$$

La permutazione $\textbf{P}$ è definita nel seguente modo:

In [102]:
P=[16,7,20,21,29,12,28,17,1,15,23,26,5,18,31,10,2,8,24,14,32,27,3,9,19,13,30,6,22,11,4,25]

data la stringa $C=(c_1,c_2,\dots,c_{32})$ allora $P(C)$ è definita nel seguente modo:

$$P(C)=(c_{16},c_7,c_{20},\dots,c_4,c_{25})$$

Gli $S$-box sono definiti nel seguente modo:

In [103]:
SBOX = [

[[14, 4, 13, 1, 2, 15, 11, 8, 3, 10, 6, 12, 5, 9, 0, 7],
 [0, 15, 7, 4, 14, 2, 13, 1, 10, 6, 12, 11, 9, 5, 3, 8],
 [4, 1, 14, 8, 13, 6, 2, 11, 15, 12, 9, 7, 3, 10, 5, 0],
 [15, 12, 8, 2, 4, 9, 1, 7, 5, 11, 3, 14, 10, 0, 6, 13],
],

[[15, 1, 8, 14, 6, 11, 3, 4, 9, 7, 2, 13, 12, 0, 5, 10],
 [3, 13, 4, 7, 15, 2, 8, 14, 12, 0, 1, 10, 6, 9, 11, 5],
 [0, 14, 7, 11, 10, 4, 13, 1, 5, 8, 12, 6, 9, 3, 2, 15],
 [13, 8, 10, 1, 3, 15, 4, 2, 11, 6, 7, 12, 0, 5, 14, 9],
],

[[10, 0, 9, 14, 6, 3, 15, 5, 1, 13, 12, 7, 11, 4, 2, 8],
 [13, 7, 0, 9, 3, 4, 6, 10, 2, 8, 5, 14, 12, 11, 15, 1],
 [13, 6, 4, 9, 8, 15, 3, 0, 11, 1, 2, 12, 5, 10, 14, 7],
 [1, 10, 13, 0, 6, 9, 8, 7, 4, 15, 14, 3, 11, 5, 2, 12],
],

[[7, 13, 14, 3, 0, 6, 9, 10, 1, 2, 8, 5, 11, 12, 4, 15],
 [13, 8, 11, 5, 6, 15, 0, 3, 4, 7, 2, 12, 1, 10, 14, 9],
 [10, 6, 9, 0, 12, 11, 7, 13, 15, 1, 3, 14, 5, 2, 8, 4],
 [3, 15, 0, 6, 10, 1, 13, 8, 9, 4, 5, 11, 12, 7, 2, 14],
],

[[2, 12, 4, 1, 7, 10, 11, 6, 8, 5, 3, 15, 13, 0, 14, 9],
 [14, 11, 2, 12, 4, 7, 13, 1, 5, 0, 15, 10, 3, 9, 8, 6],
 [4, 2, 1, 11, 10, 13, 7, 8, 15, 9, 12, 5, 6, 3, 0, 14],
 [11, 8, 12, 7, 1, 14, 2, 13, 6, 15, 0, 9, 10, 4, 5, 3],
],

[[12, 1, 10, 15, 9, 2, 6, 8, 0, 13, 3, 4, 14, 7, 5, 11],
 [10, 15, 4, 2, 7, 12, 9, 5, 6, 1, 13, 14, 0, 11, 3, 8],
 [9, 14, 15, 5, 2, 8, 12, 3, 7, 0, 4, 10, 1, 13, 11, 6],
 [4, 3, 2, 12, 9, 5, 15, 10, 11, 14, 1, 7, 6, 0, 8, 13],
],

[[4, 11, 2, 14, 15, 0, 8, 13, 3, 12, 9, 7, 5, 10, 6, 1],
 [13, 0, 11, 7, 4, 9, 1, 10, 14, 3, 5, 12, 2, 15, 8, 6],
 [1, 4, 11, 13, 12, 3, 7, 14, 10, 15, 6, 8, 0, 5, 9, 2],
 [6, 11, 13, 8, 1, 4, 10, 7, 9, 5, 0, 15, 14, 2, 3, 12],
],

[[13, 2, 8, 4, 6, 15, 11, 1, 10, 9, 3, 14, 5, 0, 12, 7],
 [1, 15, 13, 8, 10, 3, 7, 4, 12, 5, 6, 11, 0, 14, 9, 2],
 [7, 11, 4, 1, 9, 12, 14, 2, 0, 6, 10, 13, 15, 3, 5, 8],
 [2, 1, 14, 7, 4, 10, 8, 13, 15, 12, 9, 0, 3, 5, 6, 11],
]
]

Le permutazioni $\textbf{IP}$ e la sua inversa sono definite nel seguente modo:

In [104]:
IP = [58, 50, 42, 34, 26, 18, 10, 2,
      60, 52, 44, 36, 28, 20, 12, 4,
      62, 54, 46, 38, 30, 22, 14, 6,
      64, 56, 48, 40, 32, 24, 16, 8,
      57, 49, 41, 33, 25, 17, 9, 1,
      59, 51, 43, 35, 27, 19, 11, 3,
      61, 53, 45, 37, 29, 21, 13, 5,
      63, 55, 47, 39, 31, 23, 15, 7]
IP_inv= [40, 8, 48, 16, 56, 24, 64, 32,
        39, 7, 47, 15, 55, 23, 63, 31,
        38, 6, 46, 14, 54, 22, 62, 30,
        37, 5, 45, 13, 53, 21, 61, 29,
        36, 4, 44, 12, 52, 20, 60, 28,
        35, 3, 43, 11, 51, 19, 59, 27,
        34, 2, 42, 10, 50, 18, 58, 26,
        33, 1, 41, 9, 49, 17, 57, 25]

## Descrizione key schedule ##

Il key schedule è l'algoritmo che permette di generare, a partire da una chiave di 64 bit, 16 chiavi da 48 bit l'una da utilizzare nell'algoritmo del DES. È un elemento critico del sistema e pertanto deve essere un meccanismo abbastanza caotico.

L'algoritmo è il seguente:

**1)** 8 bit della chiave K vengono scartati o usati come controllo, i restanti 56 vengono sottoposti ad una prima permutazione $PC_1$;

In [105]:
PC_1 = [57, 49, 41, 33, 25, 17, 9,
        1, 58, 50, 42, 34, 26, 18,
        10, 2, 59, 51, 43, 35, 27,
        19, 11, 3, 60, 52, 44, 36,
        63, 55, 47, 39, 31, 23, 15,
        7, 62, 54, 46, 38, 30, 22,
        14, 6, 61, 53, 45, 37, 29,
        21, 13, 5, 28, 20, 12, 4]

Analizzando tale permutazione, dal fatto che in prima posizione troviamo un 57, deduciamo che il 57esimo bit della chiave inziale, diventa il primo della chiave permutata, e così via. Il quarto bit della chiave iniziale, sarà l'ultimo in quella permutata.

**2)** si divide la stringa ottenuta in due parti da 28 bit l'una, ottenendo le stringhe $C_0$ e $D_0$;

**3)** per ogni $1 \le i \le n$, $C_i$ e $D_i$ sono ottenuti a partire dai precedenti $C_{i-1}$ e $D_{i-1}$ effettuando un numero di shift a sinistra di uno o due posti, che dipende da $i$ in base al seguente schema:

N. Iterazione| N. left shifts|

1 $\rightarrow$ 1

2 $\rightarrow$ 1

3 $\rightarrow$ 2

4 $\rightarrow$ 2

5 $\rightarrow$ 2

6 $\rightarrow$ 2

7 $\rightarrow$ 2

8 $\rightarrow$ 2

9 $\rightarrow$ 1

10 $\rightarrow$ 2

11 $\rightarrow$ 2

12 $\rightarrow$ 2

13 $\rightarrow$ 2

14 $\rightarrow$ 2

15 $\rightarrow$ 2

16 $\rightarrow$ 1

Questo significa, ad esempio, che $C_3$ e $D_3$ sono ottenuti da $C_2$ e $D_2$ rispettivamente con due shift, mentre $C_9$ e $D_9$ sono ottenuti da $C_8$ e $D_8$ con un singolo shift.

**4)** Infine si applica una permutazione $PC_2$ alla stringa $C_nD_n$ per ottenere la chiave $K^n$

In [106]:
PC_2 = [14, 17, 11, 24, 1, 5, 3, 28,
        15, 6, 21, 10, 23, 19, 12, 4,
        26, 8, 16, 7, 27, 20, 13, 2,
        41, 52, 31, 37, 47, 55, 30, 40,
        51, 45, 33, 48, 44, 49, 39, 56,
        34, 53, 46, 42, 50, 36, 29, 32]

Proviamo a implementare il DES su un esempio.

Cifriamo il plaintext $x$='Schedule'

Poichè il DES ha bisogno di una stringa di 64 bit in input, trasformiamo il plaintext in esadecimali e poi in codice binario:

In [107]:
text='Schedule'

text=text.encode().hex() #passo in esadecimali

print('Il testo in esadecimali è:', text)

Il testo in esadecimali è: 5363686564756c65


In [108]:
def HexToBin(text):

    text=text.upper()

    Hex={'0000': '0', '0001': '1', '0010': '2', '0011': '3', '0100': '4','0101':'5','0110':'6', '0111': '7', '1000': '8', '1001': '9', '1010': 'A', '1011': 'B', '1100': 'C', '1101': 'D', '1110': 'E', '1111': 'F'}

    Hex2={'0': '0000', '1': '0001', '2': '0010', '3': '0011', '4': '0100','5':'0101','6':'0110', '7': '0111', '8': '1000', '9': '1001', 'A': '1010', 'B': '1011', 'C': '1100', 'D': '1101', 'E': '1110', 'F': '1111'}

    x=''

    for i in text:
        x=x+Hex2[i]

    return x

In [109]:
x=HexToBin(text)

print('il testo in binario è:',x)

print('la lunghezza del testo è',len(x), 'bit')

il testo in binario è: 0101001101100011011010000110010101100100011101010110110001100101
la lunghezza del testo è 64 bit


Implementiamo ora la funzione key schedule. Utilizzeremo come chiave la parola "computer", che trasformeremo in una stringa binaria come già visto

In [110]:
K='computer'
K=K.encode().hex()
K=HexToBin(K)

def Key_schedule(K):

    PC_1 = [57, 49, 41, 33, 25, 17, 9,
        1, 58, 50, 42, 34, 26, 18,
        10, 2, 59, 51, 43, 35, 27,
        19, 11, 3, 60, 52, 44, 36,
        63, 55, 47, 39, 31, 23, 15,
        7, 62, 54, 46, 38, 30, 22,
        14, 6, 61, 53, 45, 37, 29,
        21, 13, 5, 28, 20, 12, 4]
    PC_2 = [14, 17, 11, 24, 1, 5, 3, 28,
        15, 6, 21, 10, 23, 19, 12, 4,
        26, 8, 16, 7, 27, 20, 13, 2,
        41, 52, 31, 37, 47, 55, 30, 40,
        51, 45, 33, 48, 44, 49, 39, 56,
        34, 53, 46, 42, 50, 36, 29, 32]

    shift=[1,1,2,2,2,2,2,2,1,2,2,2,2,2,2,1]

    Key_list=[]

    #Applichiamo la prima permutazione

    Key_perm=''
    for i in PC_1:
        Key_perm=Key_perm+K[i-1]

    C=Key_perm[0:28]
    D=Key_perm[28:56]

    #Creiamo le chiavi con lo shift e la seconda permutazione

    for j in range(16):
        if shift[j]==1:
            C=C+C
            C=C[1:29]
            D=D+D
            D=D[1:29]
        else:
            C=C+C
            C=C[2:30]
            D=D+D
            D=D[2:30]

        Key_perm=C+D
        Kn=''
        for h in PC_2:
            Kn=Kn+Key_perm[h-1]

        Key_list=Key_list+[Kn]


    return Key_list


Key_list=Key_schedule(K)

print('La lista di chiavi è la seguente:')
print(Key_list)
print('la lunghezza di una chiave è:',len(Key_list[0]))

La lista di chiavi è la seguente:
['111100001011111011101110110100000000011110011000', '111000001011111011110110100101011011010010000100', '111101001111111001110110001010000000011011100101', '111001101111011101110010000110101110100010000111', '111011101101011101110111001001100100010110010001', '111011111101001101011011100010110010000101000011', '001011111101001111111011111001101100001100000000', '101111110101100111011011010100000000011101001110', '000111110101101111011011010001001001010101010100', '001111110111100111011101000010011010010011101100', '000111110110110111001101011010001101110010000001', '010110110110110110111101000010100100010000111111', '110111011010110110101101100011110101100110000000', '110100111010111010101111100000000100001101110001', '111110011011111010100110110100111000101000000100', '111100011011111000101110000000011000001001011110']
la lunghezza di una chiave è: 48


Alleggeriamo il codice della DES implementando a parte la funzione di espansione $E$ ed $f$

In [111]:
def sommaxor(a,b):

    c=''
    for i in range(len(a)):
        if a[i]==b[i]:
            c=c+'0'
        else:
            c=c+'1'

    return c

In [112]:
def espansione(text):
    SBOX = [

    [[14, 4, 13, 1, 2, 15, 11, 8, 3, 10, 6, 12, 5, 9, 0, 7],
    [0, 15, 7, 4, 14, 2, 13, 1, 10, 6, 12, 11, 9, 5, 3, 8],
    [4, 1, 14, 8, 13, 6, 2, 11, 15, 12, 9, 7, 3, 10, 5, 0],
    [15, 12, 8, 2, 4, 9, 1, 7, 5, 11, 3, 14, 10, 0, 6, 13],
    ],

    [[15, 1, 8, 14, 6, 11, 3, 4, 9, 7, 2, 13, 12, 0, 5, 10],
    [3, 13, 4, 7, 15, 2, 8, 14, 12, 0, 1, 10, 6, 9, 11, 5],
    [0, 14, 7, 11, 10, 4, 13, 1, 5, 8, 12, 6, 9, 3, 2, 15],
    [13, 8, 10, 1, 3, 15, 4, 2, 11, 6, 7, 12, 0, 5, 14, 9],
    ],

    [[10, 0, 9, 14, 6, 3, 15, 5, 1, 13, 12, 7, 11, 4, 2, 8],
    [13, 7, 0, 9, 3, 4, 6, 10, 2, 8, 5, 14, 12, 11, 15, 1],
    [13, 6, 4, 9, 8, 15, 3, 0, 11, 1, 2, 12, 5, 10, 14, 7],
    [1, 10, 13, 0, 6, 9, 8, 7, 4, 15, 14, 3, 11, 5, 2, 12],
    ],

    [[7, 13, 14, 3, 0, 6, 9, 10, 1, 2, 8, 5, 11, 12, 4, 15],
    [13, 8, 11, 5, 6, 15, 0, 3, 4, 7, 2, 12, 1, 10, 14, 9],
    [10, 6, 9, 0, 12, 11, 7, 13, 15, 1, 3, 14, 5, 2, 8, 4],
    [3, 15, 0, 6, 10, 1, 13, 8, 9, 4, 5, 11, 12, 7, 2, 14],
    ],

    [[2, 12, 4, 1, 7, 10, 11, 6, 8, 5, 3, 15, 13, 0, 14, 9],
    [14, 11, 2, 12, 4, 7, 13, 1, 5, 0, 15, 10, 3, 9, 8, 6],
    [4, 2, 1, 11, 10, 13, 7, 8, 15, 9, 12, 5, 6, 3, 0, 14],
    [11, 8, 12, 7, 1, 14, 2, 13, 6, 15, 0, 9, 10, 4, 5, 3],
    ],

    [[12, 1, 10, 15, 9, 2, 6, 8, 0, 13, 3, 4, 14, 7, 5, 11],
    [10, 15, 4, 2, 7, 12, 9, 5, 6, 1, 13, 14, 0, 11, 3, 8],
    [9, 14, 15, 5, 2, 8, 12, 3, 7, 0, 4, 10, 1, 13, 11, 6],
    [4, 3, 2, 12, 9, 5, 15, 10, 11, 14, 1, 7, 6, 0, 8, 13],
    ],

    [[4, 11, 2, 14, 15, 0, 8, 13, 3, 12, 9, 7, 5, 10, 6, 1],
    [13, 0, 11, 7, 4, 9, 1, 10, 14, 3, 5, 12, 2, 15, 8, 6],
    [1, 4, 11, 13, 12, 3, 7, 14, 10, 15, 6, 8, 0, 5, 9, 2],
    [6, 11, 13, 8, 1, 4, 10, 7, 9, 5, 0, 15, 14, 2, 3, 12],
    ],

    [[13, 2, 8, 4, 6, 15, 11, 1, 10, 9, 3, 14, 5, 0, 12, 7],
    [1, 15, 13, 8, 10, 3, 7, 4, 12, 5, 6, 11, 0, 14, 9, 2],
    [7, 11, 4, 1, 9, 12, 14, 2, 0, 6, 10, 13, 15, 3, 5, 8],
    [2, 1, 14, 7, 4, 10, 8, 13, 15, 12, 9, 0, 3, 5, 6, 11],
    ]
    ]


    E=[32,1,2,3,4,5,4,5,6,7,8,9,8,9,10,11,12,13,12,13,14,15,16,17,16,17,18,19,20,21,20,21,22,23,24,25,24,25,26,27,28,29,28,29,30,31,32,1]

    text2=''
    for i in E:
        text2=text2+text[i-1]
    return text2

def IntToBin(n):
    Bin=''
    while n>0:
        if n%2 == 0:
            Bin='0'+Bin
        else:
            Bin='1'+Bin
        n=int(n/2)
    while len(Bin) < 4:
        Bin='0'+ Bin
    return Bin


def f(A,J):

    P=[16,7,20,21,29,12,28,17,1,15,23,26,5,18,31,10,2,8,24,14,32,27,3,9,19,13,30,6,22,11,4,25]

    #espandiamo A
    A=espansione(A)

    #effettuiamo la somma xor di A con J
    B=sommaxor(A,J)

    #applichiamo gli S-Box
    C=''
    for j in range(8):
        Bj=B[6*j:6*j+6]
        r=int(Bj[0]+Bj[5],2)
        c=int(Bj[1]+Bj[2]+Bj[3]+Bj[4],2)
        C=C+IntToBin(SBOX[j][r][c])

    #applichiamo la permutazione P a C
    C1=''
    for i in P:
        C1=C1+C[i-1]
    return C1

Possiamo ora implementare il DES

In [113]:
#Processo di codifica
def DES(text,Key_list):

#text è una stringa di 64 bit
    IP = [58, 50, 42, 34, 26, 18, 10, 2,
      60, 52, 44, 36, 28, 20, 12, 4,
      62, 54, 46, 38, 30, 22, 14, 6,
      64, 56, 48, 40, 32, 24, 16, 8,
      57, 49, 41, 33, 25, 17, 9, 1,
      59, 51, 43, 35, 27, 19, 11, 3,
      61, 53, 45, 37, 29, 21, 13, 5,
      63, 55, 47, 39, 31, 23, 15, 7]
    IP_inv= [40, 8, 48, 16, 56, 24, 64, 32,
        39, 7, 47, 15, 55, 23, 63, 31,
        38, 6, 46, 14, 54, 22, 62, 30,
        37, 5, 45, 13, 53, 21, 61, 29,
        36, 4, 44, 12, 52, 20, 60, 28,
        35, 3, 43, 11, 51, 19, 59, 27,
        34, 2, 42, 10, 50, 18, 58, 26,
        33, 1, 41, 9, 49, 17, 57, 25]

    #permutazione di text tramite IP

    x=''
    for i in IP:
        x=x+text[i-1]

    L=x[:32]
    R=x[32:]
    for i in range(16):
        hold=L
        L=R
        C=f(R,Key_list[i])
        #somma xor
        R=sommaxor(C,hold)
    x=R+L

    #applichiamo IP_inv

    y=''
    for i in IP_inv:
        y=y+x[i-1]

    return y

In [114]:
#il text è Schedule in codice binario precedentemente calcolato
text='0101001101100011011010000110010101100100011101010110110001100101'
y=DES(text,Key_list)
#Mostro il codice codificato
print('y è uguale a:',y)
print('la lunghezza del ciphertext è:',len(y))

y è uguale a: 0100000110111111110100000010111010011100110010000010010111100101
la lunghezza del ciphertext è: 64


Per decifrare è necessario applicare lo stesso algoritmo

In [115]:
#definizio la funzione di decodifica
def DES_dec(text,Key_list):

    IP = [58, 50, 42, 34, 26, 18, 10, 2,
      60, 52, 44, 36, 28, 20, 12, 4,
      62, 54, 46, 38, 30, 22, 14, 6,
      64, 56, 48, 40, 32, 24, 16, 8,
      57, 49, 41, 33, 25, 17, 9, 1,
      59, 51, 43, 35, 27, 19, 11, 3,
      61, 53, 45, 37, 29, 21, 13, 5,
      63, 55, 47, 39, 31, 23, 15, 7]
    IP_inv= [40, 8, 48, 16, 56, 24, 64, 32,
        39, 7, 47, 15, 55, 23, 63, 31,
        38, 6, 46, 14, 54, 22, 62, 30,
        37, 5, 45, 13, 53, 21, 61, 29,
        36, 4, 44, 12, 52, 20, 60, 28,
        35, 3, 43, 11, 51, 19, 59, 27,
        34, 2, 42, 10, 50, 18, 58, 26,
        33, 1, 41, 9, 49, 17, 57, 25]

#permutazione di text tramite IP
    x=''
    for i in IP:
        x=x+text[i-1]

#divido x in due parti (in maniera inversa a come fatto precentemente, cioè scambiando R ed L)

    R=x[:32]
    L=x[32:]
    for i in reversed(range(16)):
        hold=R
        R=L
        C=f(L,Key_list[i])
        #somma xor
        L=sommaxor(C,hold)

    x=L+R

#applicchiamo IP_inv
    y=''
    for i in IP_inv:
        y=y+x[i-1]

    return y

In [116]:
#Testocodificato'
text='0100000110111111110100000010111010011100110010000010010111100101'
x=DES_dec(text,Key_list)
print('Il testo in chiaro è:',x)

Il testo in chiaro è: 0101001101100011011010000110010101100100011101010110110001100101


Riconvertiamo x in esadecimali e in testo in chiaro:

In [117]:
# Funzione: converto il binario in hex
def binToMyHex (text):
    # Trasformo binario in int
    decimal = int(str(text), 2)
    # Trasformo il decimale in esadecimale
    myHex = format(decimal, 'X')
    return myHex

In [125]:
#Richiamo la funzione per trasformare da binario a esadecimale
FinalHex = binToMyHex (x)
print(FinalHex)
#Stampo il testo decodificato
print(bytes.fromhex(FinalHex).decode('utf-8'))

5363686564756C65
Schedule
